**1. Multidimensional Slicing (Feature vs. Sample Extraction)**

In standard Python lists, slicing creates a copy of the data. In NumPy, slicing creates a view. Modifying a slice alters the original array. This memory-efficient design is how we extract specific batches of samples or individual feature columns cleanly.

In [2]:
import numpy as np

# Simulating a small dataset: 4 samples (rows), 3 features (columns)
# Features: [Feature A, Feature B, Feature C]
data = np.array([
    [10, 20, 30],
    [40, 50, 60],
    [70, 80, 90],
    [100, 110, 120]
], dtype=np.float32)

# Syntax: array[row_slice, column_slice]

# Pattern 1: Extract a mini-batch (e.g., first 2 rows, all features)
mini_batch = data[0:2, :]
print("First 2 samples (Rows 0 & 1):\n", mini_batch)

# Pattern 2: Extract a specific feature column (e.g., all rows, just the last feature column)
target_feature = data[:, -1]
print("\nTarget Feature column (Last Column):\n", target_feature)

# Pattern 3: Extract a specific sub-matrix grid
sub_grid = data[1:3, 0:2]
print("\nSub-grid of Rows 1-2 and Columns 0-1:\n", sub_grid)

First 2 samples (Rows 0 & 1):
 [[10. 20. 30.]
 [40. 50. 60.]]

Target Feature column (Last Column):
 [ 30.  60.  90. 120.]

Sub-grid of Rows 1-2 and Columns 0-1:
 [[40. 50.]
 [70. 80.]]


**2. Boolean Masking (Data Filtering & Outlier Removal)**

Boolean masking allows you to filter out specific numbers instantly without checking every element using standard `if` statements. When you apply a logical condition to an array, NumPy generates an identical array of `True/False` flags, which you use to index the original dataset.

In [3]:
# Imagine a matrix of raw feature scores
features = np.array([1.5, -0.2, 4.8, 99.0, -3.5, 0.5])

# Create a boolean condition mask to flag outliers or anomalies
# Let's say any feature above 10.0 or below 0.0 is invalid
invalid_mask = (features > 10.0) | (features < 0.0)
print("Boolean Mask Matrix:\n", invalid_mask)

# Extract only the valid entries using the inverse of our invalid mask (using ~)
valid_features = features[~invalid_mask]
print("\nFiltered Dataset (No outliers):\n", valid_features)

# Modifying data in-place using a mask (e.g., clipping negative values to exactly 0.0)
features[features < 0.0] = 0.0
print("\nIn-place Mutated Features (Negative values zeroed out):\n", features)

Boolean Mask Matrix:
 [False  True False  True  True False]

Filtered Dataset (No outliers):
 [1.5 4.8 0.5]

In-place Mutated Features (Negative values zeroed out):
 [ 1.5  0.   4.8 99.   0.   0.5]


**3. Fancy Indexing (Arbitrary Index Tracking & Shuffling)**

Fancy indexing refers to passing an array or list of integers to index your data. While slicing extracts contiguous blocks, fancy indexing allows you to extract non-contiguous rows or columns in any order. This is the underlying engine used to shuffle datasets during training or extract specific index sets for validation splits.

In [4]:
# A matrix representing 5 encoded data rows
dataset = np.array([
    [100, 101], # Index 0
    [200, 201], # Index 1
    [300, 301], # Index 2
    [400, 401], # Index 3
    [500, 501]  # Index 4
])

# Specify an exact list of index positions to extract in an arbitrary order
shuffled_indices = [4, 0, 2]

# Extract rows matching those index tokens instantly
sampled_rows = dataset[shuffled_indices]
print("Fancy Indexed Rows (4, 0, then 2):\n", sampled_rows)

# Fancy indexing columns explicitly
# Pull all rows, but extract the columns in reverse order [1, 0]
reversed_cols = dataset[:, [1, 0]]
print("\nReversed Feature Columns:\n", reversed_cols)

Fancy Indexed Rows (4, 0, then 2):
 [[500 501]
 [100 101]
 [300 301]]

Reversed Feature Columns:
 [[101 100]
 [201 200]
 [301 300]
 [401 400]
 [501 500]]


**4. Implementing Data Normalisation From Scratch**

In machine learning pipelines, raw features often have completely different ranges (e.g., transaction amounts up to $10,000 vs. customer ages up to 80). If you feed these unscaled numbers directly into distance-based algorithms or gradient descent loops, the large scales distort the calculations.

We will implement two fundamental scaling normalizers from scratch using vectorized NumPy arithmetic along specific axes:

**1. Min-Max Normalisation:** Scales the dataset features into a strict boundary between `0` and `1`.

$$X_{scaled} = \frac{X - X_{min}}{X_{max} - X_{min}}$$

**2. Z-Score Standardisation:** Centers features around a mean of `0` with a standard deviation of `1`.

$$X_{scaled} = \frac{X - \mu}{\sigma}$$

In [5]:
def min_max_normalize(X):
    """
    Scales each column feature of matrix X into a strict [0, 1] range.
    Uses axis=0 to compute calculations column-wise across rows.
    """
    X = np.asarray(X, dtype=np.float32)
    
    # Calculate the minimum and maximum boundary values for each feature column
    x_min = np.min(X, axis=0)
    x_max = np.max(X, axis=0)
    
    # Prevent division-by-zero crashes if a feature has zero variance
    range_delta = x_max - x_min
    range_delta[range_delta == 0.0] = 1.0
    
    # Vectorized broadcasting calculations executed instantly
    return (X - x_min) / range_delta


def z_score_standardize(X):
    """
    Standardizes each column feature to have a mean of 0 and standard deviation of 1.
    """
    X = np.asarray(X, dtype=np.float32)
    
    # Compute statistical averages column-wise
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)
    
    # Prevent division-by-zero errors for constant columns
    std[std == 0.0] = 1.0
    
    return (X - mean) / std

**5. Testing and Verifying Your Normalizers**

Run this validation cell inside your notebook to see your normalizer mathematical properties hold true.

In [6]:
# Mock Dataset: 3 samples, 2 feature metrics
# Feature 1 ranges from 10 to 30. Feature 2 ranges from 1000 to 3000.
X_raw = np.array([
    [10.0, 3000.0],
    [20.0, 1000.0],
    [30.0, 2000.0]
], dtype=np.float32)

print("Original Raw Data Matrix:\n", X_raw)

# 1. Test Min-Max Scaling
X_min_max = min_max_normalize(X_raw)
print("\n1. Min-Max Normalized Matrix (All features bounded [0, 1]):\n", X_min_max)
print("   Verified Column Minimums:", np.min(X_min_max, axis=0))
print("   Verified Column Maximums:", np.max(X_min_max, axis=0))

# 2. Test Z-Score Scaling
X_z_score = z_score_standardize(X_raw)
print("\n2. Z-Score Standardized Matrix (Mean=0, Std=1):\n", X_z_score)
print("   Verified Column Means (Should be approx 0):", np.round(np.mean(X_z_score, axis=0), 5))
print("   Verified Column Stds (Should be approx 1):", np.std(X_z_score, axis=0))

Original Raw Data Matrix:
 [[  10. 3000.]
 [  20. 1000.]
 [  30. 2000.]]

1. Min-Max Normalized Matrix (All features bounded [0, 1]):
 [[0.  1. ]
 [0.5 0. ]
 [1.  0.5]]
   Verified Column Minimums: [0. 0.]
   Verified Column Maximums: [1. 1.]

2. Z-Score Standardized Matrix (Mean=0, Std=1):
 [[-1.2247449  1.2247449]
 [ 0.        -1.2247449]
 [ 1.2247449  0.       ]]
   Verified Column Means (Should be approx 0): [0. 0.]
   Verified Column Stds (Should be approx 1): [1. 1.]
